**BITS ID** : 2025AD05248

**Name** : GOPI AGASTHIA S

**Email** : 2025ad05069@wilp.bits-pilani.ac.in

**Date** : 06.08.2026

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from datasets import load_dataset

print("="*70)
print("CNN ASSIGNMENT - DEEP NEURAL NETWORKS")
print("="*70)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("="*70)


CNN ASSIGNMENT - DEEP NEURAL NETWORKS
TensorFlow version: 2.20.0
GPU Available: True


In [3]:
# ============================================================================
# PART 1: DATASET LOADING AND EXPLORATION
# ============================================================================

print("\nLoading Cats vs Dogs dataset...")

# Load full dataset from Hugging Face
hf_dataset = load_dataset('microsoft/cats_vs_dogs', split='train')
hf_dataset = hf_dataset.shuffle(seed=42)

# Split 85% train / 15% test
n_total = len(hf_dataset)
n_train = int(n_total * 0.85)
hf_train = hf_dataset.select(range(n_train))
hf_test  = hf_dataset.select(range(n_train, n_total))

def hf_to_tf_dataset(hf_split):
    """Convert a Hugging Face split to a tf.data.Dataset of (image_tensor, label)."""
    def gen():
        for example in hf_split:
            img = example['image'].convert('RGB')
            yield np.array(img, dtype=np.uint8), example['labels']
    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
            tf.TensorSpec(shape=(),             dtype=tf.int64),
        )
    )

ds_train = hf_to_tf_dataset(hf_train)
ds_test  = hf_to_tf_dataset(hf_test)
print(f"Train samples: {n_train}  |  Test samples: {n_total - n_train}")


Loading Cats vs Dogs dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/330M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/391M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23410 [00:00<?, ? examples/s]

Train samples: 19898  |  Test samples: 3512


I0000 00:00:1786127842.344386      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [4]:
# Dataset metadata
dataset_name = "Cats vs Dogs"
dataset_source = "Hugging Face Datasets (Microsoft)"
n_samples = n_total
n_classes = 2
samples_per_class = f"min: {n_samples//2}, max: {n_samples//2}, avg: {n_samples//2}"
image_shape = [224, 224, 3]
problem_type = "binary_classification"
train_test_ratio = "85/15"
train_samples = n_train
test_samples = n_total - n_train

In [5]:
# Primary metric selection
primary_metric = "accuracy"
metric_justification = "Accuracy is chosen as the primary metric because the Cats vs Dogs dataset is balanced with approximately equal samples per class, making accuracy a reliable performance indicator."

print("\nDATASET INFORMATION")
print("-" * 70)
print(f"Dataset: {dataset_name}")
print(f"Source: {dataset_source}")
print(f"Total Samples: {n_samples}")
print(f"Number of Classes: {n_classes}")
print(f"Samples per Class: {samples_per_class}")
print(f"Image Shape: {image_shape}")
print(f"Primary Metric: {primary_metric}")
print(f"Metric Justification: {metric_justification}")
print(f"\nTrain/Test Split: {train_test_ratio}")
print(f"Training Samples: {train_samples}")
print(f"Test Samples: {test_samples}")



DATASET INFORMATION
----------------------------------------------------------------------
Dataset: Cats vs Dogs
Source: Hugging Face Datasets (Microsoft)
Total Samples: 23410
Number of Classes: 2
Samples per Class: min: 11705, max: 11705, avg: 11705
Image Shape: [224, 224, 3]
Primary Metric: accuracy
Metric Justification: Accuracy is chosen as the primary metric because the Cats vs Dogs dataset is balanced with approximately equal samples per class, making accuracy a reliable performance indicator.

Train/Test Split: 85/15
Training Samples: 19898
Test Samples: 3512


In [6]:
# Data Preprocessing
print("\nPreprocessing dataset...")

def preprocess_image(image, label):
    """Resize and normalize images"""
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment_image(image, label):
    """Apply data augmentation"""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    return image, label



Preprocessing dataset...


In [7]:
# Configure datasets
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Prepare training dataset with augmentation
ds_train = ds_train.map(preprocess_image, num_parallel_calls=AUTOTUNE)
ds_train = ds_train.map(augment_image, num_parallel_calls=AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(1000)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(AUTOTUNE)

# Prepare test dataset
ds_test = ds_test.map(preprocess_image, num_parallel_calls=AUTOTUNE)
ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(AUTOTUNE)

print("✓ Dataset preprocessing complete!")


✓ Dataset preprocessing complete!


In [8]:
# Visualize sample images
print("\nGenerating sample visualizations...")
plt.figure(figsize=(12, 8))
class_names = ['Cat', 'Dog']

for images, labels in ds_test.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(f"{class_names[labels[i].numpy()]}")
        plt.axis('off')

plt.suptitle('Sample Images from Cats vs Dogs Dataset', fontsize=16)
plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
print("✓ Saved: dataset_samples.png")
plt.close()



Generating sample visualizations...
✓ Saved: dataset_samples.png


In [9]:
# Class distribution
plt.figure(figsize=(8, 6))
class_counts = [n_samples//2, n_samples//2]
plt.bar(class_names, class_counts, color=['orange', 'skyblue'])
plt.title('Class Distribution', fontsize=14)
plt.ylabel('Number of Images')
plt.xlabel('Class')
for i, v in enumerate(class_counts):
    plt.text(i, v + 100, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
print("✓ Saved: class_distribution.png")
plt.close()


✓ Saved: class_distribution.png


In [10]:
# ============================================================================
# PART 2: CUSTOM CNN IMPLEMENTATION
# ============================================================================

def build_custom_cnn(input_shape, n_classes):
    """
    Build custom CNN architecture with Global Average Pooling

    Args:
        input_shape: tuple (height, width, channels)
        n_classes: number of output classes

    Returns:
        model: compiled CNN model
    """
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),

        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Global Average Pooling (MANDATORY - NO Flatten+Dense)
        layers.GlobalAveragePooling2D(),

        # Output layer
        layers.Dense(1, activation='sigmoid') if n_classes == 2 else layers.Dense(n_classes, activation='softmax')
    ], name='Custom_CNN')

    return model


In [11]:

# Create model instance
print("\nBuilding Custom CNN architecture...")
custom_cnn = build_custom_cnn(image_shape, n_classes)

# Compile model
custom_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nCUSTOM CNN ARCHITECTURE")
print("-" * 70)
custom_cnn.summary()


Building Custom CNN architecture...

CUSTOM CNN ARCHITECTURE
----------------------------------------------------------------------


Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 288,929 (1.10 MB)

 Trainable params: 288,033 (1.10 MB)

 Non-trainable params: 896 (3.50 KB)

In [12]:
# Count architecture components
conv_layers = len([layer for layer in custom_cnn.layers if isinstance(layer, layers.Conv2D)])
pooling_layers = len([layer for layer in custom_cnn.layers if isinstance(layer, (layers.MaxPooling2D, layers.AveragePooling2D))])
has_gap = any(isinstance(layer, layers.GlobalAveragePooling2D) for layer in custom_cnn.layers)
custom_cnn_total_params = custom_cnn.count_params()

print(f"\nArchitecture Summary:")
print(f"Conv2D Layers: {conv_layers}")
print(f"Pooling Layers: {pooling_layers}")
print(f"Has Global Average Pooling: {has_gap}")
print(f"Total Parameters: {custom_cnn_total_params:,}")


Architecture Summary:
Conv2D Layers: 6
Pooling Layers: 3
Has Global Average Pooling: True
Total Parameters: 288,929


In [13]:
# Train Custom CNN

EPOCHS = 20

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

# Track training time
custom_cnn_start_time = time.time()

# Train model
history_custom = custom_cnn.fit(
    ds_train,
    epochs=EPOCHS,
    validation_data=ds_test,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

custom_cnn_training_time = time.time() - custom_cnn_start_time

Epoch 1/20
      2/Unknown 18s 73ms/step - accuracy: 0.5703 - loss: 0.8934

I0000 00:00:1786127864.493636     152 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


622/622 ━━━━━━━━━━━━━━━━━━━━ 99s 131ms/step - accuracy: 0.6274 - loss: 0.6402 - val_accuracy: 0.5999 - val_loss: 0.7318 - learning_rate: 0.0010
Epoch 2/20
622/622 ━━━━━━━━━━━━━━━━━━━━ 45s 73ms/step - accuracy: 0.6914 - loss: 0.5806 - val_accuracy: 0.7013 - val_loss: 0.5889 - learning_rate: 0.0010
Epoch 3/20
622/622 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.7426 - loss: 0.5236 - val_accuracy: 0.5578 - val_loss: 1.0768 - learning_rate: 0.0010
Epoch 4/20
622/622 ━━━━━━━━━━━━━━━━━━━━ 45s 73ms/step - accuracy: 0.7878 - loss: 0.4532 - val_accuracy: 0.6412 - val_loss: 0.8110 - learning_rate: 0.0010
Epoch 5/20
622/622 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.8366 - loss: 0.3726 - val_accuracy: 0.8776 - val_loss: 0.3174 - learning_rate: 0.0010
Epoch 6/20
622/622 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.8782 - loss: 0.2943 - val_accuracy: 0.7964 - val_loss: 0.4892 - learning_rate: 0.0010
Epoch 7/20
622/622 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.8962 - loss: 0.249

In [14]:
custom_cnn_training_time = time.time() - custom_cnn_start_time

# Track initial and final loss
custom_cnn_initial_loss = history_custom.history['loss'][0]
custom_cnn_final_loss = history_custom.history['loss'][-1]

print(f"\n✓ Training completed in {custom_cnn_training_time:.2f} seconds")
print(f"Initial Loss: {custom_cnn_initial_loss:.4f}")
print(f"Final Loss: {custom_cnn_final_loss:.4f}")
print(f"Loss Reduction: {((custom_cnn_initial_loss - custom_cnn_final_loss) / custom_cnn_initial_loss * 100):.2f}%")



✓ Training completed in 956.96 seconds
Initial Loss: 0.6402
Final Loss: 0.0746
Loss Reduction: 88.35%


In [15]:
# Evaluate Custom CNN

# Make predictions on test set
y_pred_probs = custom_cnn.predict(ds_test, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

# Get true labels
y_test = np.concatenate([y for x, y in ds_test], axis=0)

# Calculate all 4 required metrics
custom_cnn_accuracy = accuracy_score(y_test, y_pred)
custom_cnn_precision = precision_score(y_test, y_pred, average='macro')
custom_cnn_recall = recall_score(y_test, y_pred, average='macro')
custom_cnn_f1 = f1_score(y_test, y_pred, average='macro')

print("\nCustom CNN Performance:")
print(f"Accuracy:  {custom_cnn_accuracy:.4f}")
print(f"Precision: {custom_cnn_precision:.4f}")
print(f"Recall:    {custom_cnn_recall:.4f}")
print(f"F1-Score:  {custom_cnn_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))


Custom CNN Performance:
Accuracy:  0.9544
Precision: 0.9545
Recall:    0.9544
F1-Score:  0.9544

Classification Report:
              precision    recall  f1-score   support

         Cat       0.96      0.95      0.95      1744
         Dog       0.95      0.96      0.95      1768

    accuracy                           0.95      3512
   macro avg       0.95      0.95      0.95      3512
weighted avg       0.95      0.95      0.95      3512



In [16]:

# Visualize Custom CNN Results
print("\nGenerating Custom CNN visualizations...")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_custom.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history_custom.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Custom CNN - Loss Curve', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_custom.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history_custom.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Custom CNN - Accuracy Curve', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('custom_cnn_training_curves.png', dpi=150, bbox_inches='tight')
print("✓ Saved: custom_cnn_training_curves.png")
plt.close()



Generating Custom CNN visualizations...
✓ Saved: custom_cnn_training_curves.png


In [17]:

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.title('Custom CNN - Confusion Matrix', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('custom_cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Saved: custom_cnn_confusion_matrix.png")
plt.close()


✓ Saved: custom_cnn_confusion_matrix.png


In [34]:
# ============================================================================
# PART 3: TRANSFER LEARNING IMPLEMENTATION
# ============================================================================
from tensorflow.keras.applications.resnet50 import preprocess_input

pretrained_model_name = "ResNet50"

def build_transfer_learning_model(base_model_name, input_shape, n_classes):
    """
    Build transfer learning model with Global Average Pooling

    Args:
        base_model_name: string (ResNet50)
        input_shape: tuple (height, width, channels)
        n_classes: number of output classes

    Returns:
        model: compiled transfer learning model
        base_model: the base model for layer counting
    """
    # Load pre-trained ResNet50 without top layers
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # Freeze base layers
    base_model.trainable = False

    # Build model with Global Average Pooling
    inputs = tf.keras.Input(shape=input_shape)
    x = preprocess_input(inputs)          # ← Fix #2: ResNet50-specific normalization
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1 if n_classes == 2 else n_classes,
                           activation='sigmoid' if n_classes == 2 else 'softmax')(x)

    model = tf.keras.Model(inputs, outputs, name='Transfer_Learning_ResNet50')
    return model, base_model

In [35]:
# Create transfer learning model
print("\nBuilding Transfer Learning model...")
transfer_model, base_model = build_transfer_learning_model(pretrained_model_name, image_shape, n_classes)

# Compile model
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Count layers and parameters
frozen_layers = len([layer for layer in base_model.layers if not layer.trainable])
trainable_layers = len([layer for layer in transfer_model.layers if layer.trainable])
total_parameters = transfer_model.count_params()
trainable_parameters = sum([tf.size(var).numpy() for var in transfer_model.trainable_variables])

print(f"\nBase Model: {pretrained_model_name}")
print(f"Frozen Layers: {frozen_layers}")
print(f"Trainable Layers: {trainable_layers}")
print(f"Total Parameters: {total_parameters:,}")
print(f"Trainable Parameters: {trainable_parameters:,}")
print(f"Using Global Average Pooling: YES")

print("\nModel Architecture:")
print("-" * 70)
transfer_model.summary()


Building Transfer Learning model...

Base Model: ResNet50
Frozen Layers: 175
Trainable Layers: 4
Total Parameters: 23,589,761
Trainable Parameters: 2,049
Using Global Average Pooling: YES

Model Architecture:
----------------------------------------------------------------------


Model: "Transfer_Learning_ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ input_layer_8[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ input_layer_8[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ input_layer_8[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │      2,049 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [36]:
# Train Transfer Learning Model

# Training configuration
tl_learning_rate = 0.001
tl_epochs = 10
tl_batch_size = 32
tl_optimizer = "Adam"

# Callbacks
early_stopping_tl = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Track training time
tl_start_time = time.time()

# Train model
history_tl_phase1 = transfer_model.fit(
    ds_train,
    epochs=5,
    validation_data=ds_test,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                             restore_best_weights=True)],
    verbose=1
)

base_model.trainable = True
fine_tune_at = 165  # freeze everything before this layer index

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile at 100× lower LR to avoid destroying pre-trained weights
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_tl = transfer_model.fit(
    ds_train,
    epochs=15,
    validation_data=ds_test,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                             restore_best_weights=True)],
    verbose=1
)

tl_training_time = time.time() - tl_start_time

# Track initial and final loss
tl_initial_loss = history_tl.history['loss'][0]
tl_final_loss = history_tl.history['loss'][-1]

print(f"\n✓ Training completed in {tl_training_time:.2f} seconds")
print(f"Initial Loss: {tl_initial_loss:.4f}")
print(f"Final Loss: {tl_final_loss:.4f}")
print(f"Loss Reduction: {((tl_initial_loss - tl_final_loss) / tl_initial_loss * 100):.2f}%")


Epoch 1/5
622/622 ━━━━━━━━━━━━━━━━━━━━ 53s 69ms/step - accuracy: 0.5591 - loss: 0.6828 - val_accuracy: 0.6375 - val_loss: 0.6564
Epoch 2/5
622/622 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.5993 - loss: 0.6622 - val_accuracy: 0.6384 - val_loss: 0.6466
Epoch 3/5
180/622 ━━━━━━━━━━━━━━━━━━━━ 1:59 270ms/step - accuracy: 0.6099 - loss: 0.6559

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


622/622 ━━━━━━━━━━━━━━━━━━━━ 74s 118ms/step - accuracy: 0.6098 - loss: 0.6564 - val_accuracy: 0.6444 - val_loss: 0.6439
Epoch 4/5
622/622 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.6223 - loss: 0.6500 - val_accuracy: 0.6518 - val_loss: 0.6360
Epoch 5/5
622/622 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.6213 - loss: 0.6476 - val_accuracy: 0.6640 - val_loss: 0.6307
Epoch 1/15
622/622 ━━━━━━━━━━━━━━━━━━━━ 60s 77ms/step - accuracy: 0.6573 - loss: 0.6372 - val_accuracy: 0.7229 - val_loss: 0.5546
Epoch 2/15
622/622 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.7075 - loss: 0.5711 - val_accuracy: 0.7369 - val_loss: 0.5347
Epoch 3/15
622/622 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.7263 - loss: 0.5436 - val_accuracy: 0.7397 - val_loss: 0.5245
Epoch 4/15
622/622 ━━━━━━━━━━━━━━━━━━━━ 37s 59ms/step - accuracy: 0.7352 - loss: 0.5277 - val_accuracy: 0.7588 - val_loss: 0.5038
Epoch 5/15
622/622 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.7466 - loss: 0.5123 - val_accuracy

In [37]:
# Evaluate Transfer Learning Model

# Make predictions on test set
y_pred_tl_probs = transfer_model.predict(ds_test, verbose=0)
y_pred_tl = (y_pred_tl_probs > 0.5).astype(int).flatten()

# Calculate all 4 required metrics
tl_accuracy = accuracy_score(y_test, y_pred_tl)
tl_precision = precision_score(y_test, y_pred_tl, average='macro')
tl_recall = recall_score(y_test, y_pred_tl, average='macro')
tl_f1 = f1_score(y_test, y_pred_tl, average='macro')

print("\nTransfer Learning Performance:")
print(f"Accuracy:  {tl_accuracy:.4f}")
print(f"Precision: {tl_precision:.4f}")
print(f"Recall:    {tl_recall:.4f}")
print(f"F1-Score:  {tl_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tl, target_names=['Cat', 'Dog']))



Transfer Learning Performance:
Accuracy:  0.7668
Precision: 0.7673
Recall:    0.7666
F1-Score:  0.7666

Classification Report:
              precision    recall  f1-score   support

         Cat       0.78      0.74      0.76      1744
         Dog       0.76      0.79      0.77      1768

    accuracy                           0.77      3512
   macro avg       0.77      0.77      0.77      3512
weighted avg       0.77      0.77      0.77      3512



In [38]:
# Visualize Transfer Learning Results
print("\nGenerating Transfer Learning visualizations...")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_tl.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history_tl.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Transfer Learning - Loss Curve', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_tl.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history_tl.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Transfer Learning - Accuracy Curve', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('transfer_learning_training_curves.png', dpi=150, bbox_inches='tight')
print("✓ Saved: transfer_learning_training_curves.png")
plt.close()


Generating Transfer Learning visualizations...
✓ Saved: transfer_learning_training_curves.png


In [39]:
# Confusion Matrix
cm_tl = confusion_matrix(y_test, y_pred_tl)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens', xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.title('Transfer Learning - Confusion Matrix', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('transfer_learning_confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Saved: transfer_learning_confusion_matrix.png")
plt.close()

✓ Saved: transfer_learning_confusion_matrix.png


In [40]:

# ============================================================================
# PART 4: MODEL COMPARISON
# ============================================================================

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Training Time (s)', 'Parameters'],
    'Custom CNN': [
        f"{custom_cnn_accuracy:.4f}",
        f"{custom_cnn_precision:.4f}",
        f"{custom_cnn_recall:.4f}",
        f"{custom_cnn_f1:.4f}",
        f"{custom_cnn_training_time:.2f}",
        f"{custom_cnn_total_params:,}"
    ],
    'Transfer Learning': [
        f"{tl_accuracy:.4f}",
        f"{tl_precision:.4f}",
        f"{tl_recall:.4f}",
        f"{tl_f1:.4f}",
        f"{tl_training_time:.2f}",
        f"{trainable_parameters:,}"
    ]
})

print("\n" + comparison_df.to_string(index=False))


           Metric Custom CNN Transfer Learning
         Accuracy     0.9544            0.7668
        Precision     0.9545            0.7673
           Recall     0.9544            0.7666
         F1-Score     0.9544            0.7666
Training Time (s)     956.96            621.19
       Parameters    288,929             2,049


In [41]:
# Visual Comparison
print("\nGenerating comparison visualizations...")

# Metrics comparison
metrics_data = {
    'Accuracy': [custom_cnn_accuracy, tl_accuracy],
    'Precision': [custom_cnn_precision, tl_precision],
    'Recall': [custom_cnn_recall, tl_recall],
    'F1-Score': [custom_cnn_f1, tl_f1]
}

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metrics_data))
width = 0.35

custom_values = [metrics_data[m][0] for m in metrics_data]
tl_values = [metrics_data[m][1] for m in metrics_data]

bars1 = ax.bar(x - width/2, custom_values, width, label='Custom CNN', color='skyblue')
bars2 = ax.bar(x + width/2, tl_values, width, label='Transfer Learning', color='lightgreen')

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_data.keys())
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Saved: model_comparison.png")
plt.close()



Generating comparison visualizations...
✓ Saved: model_comparison.png


In [45]:
# ============================================================================
# PART 5: ANALYSIS
# ============================================================================

analysis_text = f"""The custom CNN outperformed the ResNet50 transfer-learning model, achieving {custom_cnn_accuracy:.1%} accuracy compared with {tl_accuracy:.1%}, a {(custom_cnn_accuracy - tl_accuracy)*100:.1f} percentage-point difference. Although ResNet50 provides powerful ImageNet-pretrained features, its performance was limited because most layers were frozen, with only {trainable_parameters:,} of {total_parameters:,} parameters trainable. This restricted the model's ability to adapt to the Cats vs Dogs dataset.
The custom CNN required {custom_cnn_training_time:.0f} seconds for {EPOCHS} epochs, while ResNet50 required {tl_training_time:.0f} seconds for {tl_epochs} epochs. Despite faster training, ResNet50 achieved substantially lower performance. The custom CNN also achieved a lower final loss ({custom_cnn_final_loss:.4f}) than transfer learning ({tl_final_loss:.4f}), indicating better task-specific learning.
Global Average Pooling reduced parameters before classification while retaining spatial feature information. Overall, the custom CNN was the preferred model, achieving higher accuracy, precision, recall, and F1-score. ResNet50 could potentially improve through greater fine-tuning, optimized learning rates, or a more suitable pre-trained architecture."""

print("\n" + analysis_text)
print(f"\nAnalysis word count: {len(analysis_text.split())} words")
if len(analysis_text.split()) > 200:
    print("⚠ Warning: Analysis exceeds 200 words (guideline)")
else:
    print("✓ Analysis within word count guideline")


The custom CNN outperformed the ResNet50 transfer-learning model, achieving 95.4% accuracy compared with 76.7%, a 18.8 percentage-point difference. Although ResNet50 provides powerful ImageNet-pretrained features, its performance was limited because most layers were frozen, with only 2,049 of 23,589,761 parameters trainable. This restricted the model's ability to adapt to the Cats vs Dogs dataset.
The custom CNN required 957 seconds for 20 epochs, while ResNet50 required 621 seconds for 10 epochs. Despite faster training, ResNet50 achieved substantially lower performance. The custom CNN also achieved a lower final loss (0.0746) than transfer learning (0.4572), indicating better task-specific learning.
Global Average Pooling reduced parameters before classification while retaining spatial feature information. Overall, the custom CNN was the preferred model, achieving higher accuracy, precision, recall, and F1-score. ResNet50 could potentially improve through greater fine-tuning, optimi

In [46]:
# ============================================================================
# PART 6: ASSIGNMENT RESULTS SUMMARY (JSON OUTPUT)
# ============================================================================

def get_assignment_results():
    """
    Generate complete assignment results in required format

    Returns:
        dict: Complete results with all required fields
    """
    framework_used = "keras"

    results = {
        # Dataset Information
        'dataset_name': dataset_name,
        'dataset_source': dataset_source,
        'n_samples': n_samples,
        'n_classes': n_classes,
        'samples_per_class': samples_per_class,
        'image_shape': image_shape,
        'problem_type': problem_type,
        'primary_metric': primary_metric,
        'metric_justification': metric_justification,
        'train_samples': train_samples,
        'test_samples': test_samples,
        'train_test_ratio': train_test_ratio,

        # Custom CNN Results
        'custom_cnn': {
            'framework': framework_used,
            'architecture': {
                'conv_layers': conv_layers,
                'pooling_layers': pooling_layers,
                'has_global_average_pooling': True,
                'output_layer': 'sigmoid',
                'total_parameters': int(custom_cnn_total_params)
            },
            'training_config': {
                'learning_rate': 0.001,
                'n_epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'optimizer': 'Adam',
                'loss_function': 'binary_crossentropy'
            },
            'initial_loss': float(custom_cnn_initial_loss),
            'final_loss': float(custom_cnn_final_loss),
            'training_time_seconds': float(custom_cnn_training_time),
            'accuracy': float(custom_cnn_accuracy),
            'precision': float(custom_cnn_precision),
            'recall': float(custom_cnn_recall),
            'f1_score': float(custom_cnn_f1)
        },

        # Transfer Learning Results
        'transfer_learning': {
            'framework': framework_used,
            'base_model': pretrained_model_name,
            'frozen_layers': frozen_layers,
            'trainable_layers': trainable_layers,
            'has_global_average_pooling': True,
            'total_parameters': int(total_parameters),
            'trainable_parameters': int(trainable_parameters),
            'training_config': {
                'learning_rate': tl_learning_rate,
                'n_epochs': tl_epochs,
                'batch_size': tl_batch_size,
                'optimizer': tl_optimizer,
                'loss_function': 'binary_crossentropy'
            },
            'initial_loss': float(tl_initial_loss),
            'final_loss': float(tl_final_loss),
            'training_time_seconds': float(tl_training_time),
            'accuracy': float(tl_accuracy),
            'precision': float(tl_precision),
            'recall': float(tl_recall),
            'f1_score': float(tl_f1)
        },

        # Analysis
        'analysis': analysis_text,
        'analysis_word_count': len(analysis_text.split()),

        # Training Success Indicators
        'custom_cnn_loss_decreased': custom_cnn_final_loss < custom_cnn_initial_loss,
        'transfer_learning_loss_decreased': tl_final_loss < tl_initial_loss,
    }

    return results

try:
    assignment_results = get_assignment_results()

    print("\nASSIGNMENT RESULTS JSON:")
    print("-" * 70)
    print(json.dumps(assignment_results, indent=2))

    # Save to file
    with open('assignment_results.json', 'w') as f:
        json.dump(assignment_results, f, indent=2)
    print("\n✓ Saved: assignment_results.json")

except Exception as e:
    print(f"\n⚠ ERROR generating results: {str(e)}")
    print("Please ensure all variables are properly defined")



ASSIGNMENT RESULTS JSON:
----------------------------------------------------------------------
{
  "dataset_name": "Cats vs Dogs",
  "dataset_source": "Hugging Face Datasets (Microsoft)",
  "n_samples": 23410,
  "n_classes": 2,
  "samples_per_class": "min: 11705, max: 11705, avg: 11705",
  "image_shape": [
    224,
    224,
    3
  ],
  "problem_type": "binary_classification",
  "primary_metric": "accuracy",
  "metric_justification": "Accuracy is chosen as the primary metric because the Cats vs Dogs dataset is balanced with approximately equal samples per class, making accuracy a reliable performance indicator.",
  "train_samples": 19898,
  "test_samples": 3512,
  "train_test_ratio": "85/15",
  "custom_cnn": {
    "framework": "keras",
    "architecture": {
      "conv_layers": 6,
      "pooling_layers": 3,
      "has_global_average_pooling": true,
      "output_layer": "sigmoid",
      "total_parameters": 288929
    },
    "training_config": {
      "learning_rate": 0.001,
      "n_